In [0]:
# =============================================================================
# GOLD LAYER - ANALYTICS AGGREGATIONS
# =============================================================================
# Purpose: Create analytics-ready aggregations and business metrics
# Input: cryptocatalog.crypto_refined.silver_ohlcv_1min
# Outputs:
#   - gold_price_summary_1h (hourly aggregations)
#   - gold_price_summary_1d (daily aggregations)
#   - gold_technical_indicators (moving averages)
#   - gold_alerts (price & volume alerts)
#   - gold_alerts_15 (15-minute volume alerts)
# =============================================================================

from pyspark.sql import functions as F, Window
from pyspark.sql.types import *

# Configuration
catalog_name = "cryptocatalog"
silver_table = f"{catalog_name}.crypto_refined.silver_ohlcv_1min"
gold_hourly_table = f"{catalog_name}.crypto_analytics.gold_price_summary_1h"
gold_daily_table = f"{catalog_name}.crypto_analytics.gold_price_summary_1d"
gold_indicators_table = f"{catalog_name}.crypto_analytics.gold_technical_indicators"
gold_alerts_table = f"{catalog_name}.crypto_analytics.gold_alerts"
gold_15_alerts_table = f"{catalog_name}.crypto_analytics.gold_alerts_15"

print("🥇 GOLD LAYER CONFIGURATION")
print("=" * 80)
print(f"Input Table:  {silver_table}")
print(f"\nOutput Tables:")
print(f"  • {gold_hourly_table}")
print(f"  • {gold_daily_table}")
print(f"  • {gold_indicators_table}")
print(f"  • {gold_alerts_table}")
print(f"  • {gold_15_alerts_table}")
print("=" * 80)

# Create Unity Catalog schemas if not exists
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.crypto_analytics")
    spark.sql(f"ALTER SCHEMA {catalog_name}.crypto_analytics SET DBPROPERTIES ('layer' = 'gold', 'description' = 'Analytics-ready crypto metrics')")
    print(f"\n✅ Schema ready: {catalog_name}.crypto_analytics")
except Exception as e:
    print(f"⚠️ Schema setup: {e}")

In [0]:
def create_gold_hourly_summary():
    """
    Gold Layer: Hourly price aggregations
    - Aggregates 1-minute candles into hourly OHLCV
    - Includes volume, volatility, and price change metrics
    """
    print("🥇 Creating hourly price summary...")
    
    df_silver = spark.table(silver_table)
    
    df_hourly = df_silver \
        .withColumn("hour", F.date_trunc("hour", F.col("timestamp"))) \
        .groupBy("hour", "symbol", "quote_currency") \
        .agg(
            F.first("open_price").alias("open"),
            F.max("high_price").alias("high"),
            F.min("low_price").alias("low"),
            F.last("close_price").alias("close"),
            F.sum("volume_btc").alias("total_volume_btc"),
            F.sum("volume_usd").alias("total_volume_usd"),
            F.count("*").alias("num_candles"),
            F.avg("close_price").alias("avg_price"),
            F.stddev("close_price").alias("price_volatility")
        ) \
        .withColumn("price_change", F.col("close") - F.col("open")) \
        .withColumn("price_change_pct", (F.col("close") - F.col("open")) / F.col("open") * 100) \
        .withColumn("created_at", F.current_timestamp())
    
    df_hourly.write.format("delta").mode("overwrite").saveAsTable(gold_hourly_table)
    hourly_count = df_hourly.count()
    print(f"  ✅ Created {gold_hourly_table} ({hourly_count:,} records)")
    return df_hourly

df_hourly = create_gold_hourly_summary()

In [0]:
def create_gold_daily_summary():
    """
    Gold Layer: Daily price aggregations
    - Aggregates 1-minute candles into daily OHLCV
    - Includes intraday high/low and volume metrics
    """
    print("🥇 Creating daily price summary...")
    
    df_silver = spark.table(silver_table)
    
    df_daily = df_silver \
        .groupBy("date", "symbol", "quote_currency") \
        .agg(
            F.first("open_price").alias("open"),
            F.max("high_price").alias("high"),
            F.min("low_price").alias("low"),
            F.last("close_price").alias("close"),
            F.sum("volume_btc").alias("total_volume_btc"),
            F.sum("volume_usd").alias("total_volume_usd"),
            F.count("*").alias("num_candles"),
            F.avg("close_price").alias("avg_price"),
            F.stddev("close_price").alias("price_volatility"),
            F.max("close_price").alias("intraday_high"),
            F.min("close_price").alias("intraday_low")
        ) \
        .withColumn("price_change", F.col("close") - F.col("open")) \
        .withColumn("price_change_pct", (F.col("close") - F.col("open")) / F.col("open") * 100) \
        .withColumn("created_at", F.current_timestamp())
    
    df_daily.write.format("delta").mode("overwrite").saveAsTable(gold_daily_table)
    daily_count = df_daily.count()
    print(f"  ✅ Created {gold_daily_table} ({daily_count:,} records)")
    return df_daily

df_daily = create_gold_daily_summary()

In [0]:
def create_gold_technical_indicators():
    """
    Gold Layer: Technical indicators (Moving Averages)
    - MA 7, 25, 99 (short, medium, long-term trends)
    - Volume MA and volatility
    - Trading signals (bullish/bearish/neutral)
    """
    print("🥇 Creating technical indicators...")
    
    df_silver = spark.table(silver_table).orderBy("timestamp")
    
    # Window specs
    window_7 = Window.orderBy("timestamp").rowsBetween(-6, 0)
    window_25 = Window.orderBy("timestamp").rowsBetween(-24, 0)
    window_99 = Window.orderBy("timestamp").rowsBetween(-98, 0)
    
    df_indicators = df_silver.select(
        "timestamp", "symbol", "quote_currency", "close_price",
        F.avg("close_price").over(window_7).alias("ma_7"),
        F.avg("close_price").over(window_25).alias("ma_25"),
        F.avg("close_price").over(window_99).alias("ma_99"),
        F.avg("volume_usd").over(window_7).alias("vol_ma_7"),
        F.stddev("close_price").over(window_7).alias("volatility_7")
    ).withColumn("ma_signal", 
        F.when((F.col("ma_7") > F.col("ma_25")) & (F.col("ma_25") > F.col("ma_99")), "bullish")
        .when((F.col("ma_7") < F.col("ma_25")) & (F.col("ma_25") < F.col("ma_99")), "bearish")
        .otherwise("neutral")
    ).withColumn("created_at", F.current_timestamp())
    
    df_indicators.write.format("delta").mode("overwrite").saveAsTable(gold_indicators_table)
    indicators_count = df_indicators.count()
    print(f"  ✅ Created {gold_indicators_table} ({indicators_count:,} records)")
    return df_indicators

df_indicators = create_gold_technical_indicators()

In [0]:
def create_gold_alerts():
    """
    Gold Layer: Price and volume alerts
    - HIGH_VOLATILITY: >2% price change
    - HIGH_VOLUME: >$5M volume
    - PRICE_SURGE: >1% increase
    - PRICE_DROP: >1% decrease
    """
    print("🥇 Creating alert table...")
    
    df_silver = spark.table(silver_table)
    
    df_alerts = df_silver.select(
        "timestamp", "symbol", "quote_currency",
        "close_price", "price_change", "price_change_pct",
        "volume_usd",
        F.when(F.abs(F.col("price_change_pct")) > 2, "HIGH_VOLATILITY")
        .when(F.col("volume_usd") > 5000000, "HIGH_VOLUME")
        .when(F.col("price_change_pct") > 1, "PRICE_SURGE")
        .when(F.col("price_change_pct") < -1, "PRICE_DROP")
        .otherwise(None).alias("alert_type")
    ).filter(F.col("alert_type").isNotNull()) \
    .withColumn("created_at", F.current_timestamp())
    
    df_alerts.write.format("delta").mode("overwrite").saveAsTable(gold_alerts_table)
    alerts_count = df_alerts.count()
    print(f"  ✅ Created {gold_alerts_table} ({alerts_count:,} records)")
    return df_alerts

df_alerts = create_gold_alerts()

In [0]:
def create_gold_15_summary():
    """
    Gold Layer: 15-minute volume alerts
    - Detects volume spikes >0.2% change from previous 15min window
    """
    print("🥇 Creating 15-minute volume alerts...")
    
    df_min = spark.table(silver_table)
    
    df_15 = df_min.withColumn("15Min", F.window(F.col("timestamp"), "15 minutes").start) \
        .groupBy("15Min", "symbol", "quote_currency") \
        .agg(F.avg("volume_usd").alias("avg_15"))
    
    df_15_alert = df_15.withColumn(
        "avg_15_prev", 
        F.lag("avg_15", 1).over(Window.partitionBy("symbol", "quote_currency").orderBy("15Min"))
    ).withColumn(
        "perc_chg",
        F.when(
            (F.col("avg_15_prev").isNotNull()) & (F.col("avg_15_prev") != 0),
            ((F.col("avg_15") - F.col("avg_15_prev")) / F.col("avg_15_prev") * 100)
        ).otherwise(None)
    ).withColumn(
        "Alert_15min",
        F.when(F.abs(F.col("perc_chg")) > 0.2, True).otherwise(False)
    )
    
    df_15_alert.write.format("delta").mode("overwrite").saveAsTable(gold_15_alerts_table)
    alerts_15_count = df_15_alert.count()
    print(f"  ✅ Created {gold_15_alerts_table} ({alerts_15_count:,} records)")
    return df_15_alert

df_alerts_15 = create_gold_15_summary()

In [0]:
# Display summary of all gold tables
print("\n" + "="*80)
print("✅ GOLD LAYER COMPLETE")
print("="*80)

print("\n📊 Hourly Summary Sample:")
df_hourly.orderBy(F.desc("hour")).limit(5).show(truncate=False)

print("\n📊 Daily Summary Sample:")
df_daily.orderBy(F.desc("date")).limit(5).show(truncate=False)

print("\n📊 Technical Indicators Sample:")
df_indicators.orderBy(F.desc("timestamp")).limit(5).show(truncate=False)

print("\n🚨 Recent Alerts:")
df_alerts.orderBy(F.desc("timestamp")).limit(10).show(truncate=False)